In [9]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split, cross_validate
from surprise import accuracy

ratings = pd.read_csv(
    "../data/ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"}
)

movies = pd.read_csv(
    "../data/movies.csv",
    dtype={"movieId": "int32"}
)

# Filter to users and movies with enough ratings 
min_user_ratings = 20
min_movie_ratings = 20

user_counts = ratings['userId'].value_counts()
movie_counts = ratings['movieId'].value_counts()

active_users = user_counts[user_counts >= min_user_ratings].index
popular_movies = movie_counts[movie_counts >= min_movie_ratings].index

filtered = ratings[
    ratings['userId'].isin(active_users) & ratings['movieId'].isin(popular_movies)
]

print(f"Original: {len(ratings):,} ratings")
print(f"After filtering (min {min_user_ratings} ratings/user, {min_movie_ratings}/movie): {len(filtered):,} ratings")

# Further random sample if still too large for your machine
SAMPLE_SIZE = 2_000_000
if len(filtered) > SAMPLE_SIZE:
    filtered = filtered.sample(n=SAMPLE_SIZE, random_state=42)
    print(f"Randomly sampled down to: {len(filtered):,} ratings")

Original: 32,000,204 ratings
After filtering (min 20 ratings/user, 20/movie): 31,725,920 ratings
Randomly sampled down to: 2,000,000 ratings


In [2]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(filtered[['userId', 'movieId', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)    

In [3]:
model = SVD(n_factors=100, random_state=42)
model.fit(trainset)

predictions = model.test(testset)

print(f"RMSE: {accuracy.rmse(predictions):.4f}")
print(f"MAE: {accuracy.mae(predictions):.4f}")

RMSE: 0.9049
RMSE: 0.9049
MAE:  0.6931
MAE: 0.6931


In [4]:
cv_results = cross_validate(model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9056  0.9031  0.9044  0.9052  0.9049  0.9047  0.0009  
MAE (testset)     0.6932  0.6919  0.6928  0.6929  0.6936  0.6929  0.0006  
Fit time          41.08   41.89   42.07   41.91   41.31   41.65   0.38    
Test time         6.21    8.52    9.43    8.33    7.31    7.96    1.10    


In [10]:
def get_top_n_recommendations(model, user_id, ratings_df, movies_df, n=10):
    # ...rated by this user
    rated_movies = ratings_df[ratings_df['userId'] == user_id]['movieId'].unique()

    # movie that hasn't been rated yet
    all_movies = movies_df['movieId'].unique()
    unrated_movies = [m for m in all_movies if m not in rated_movies]

    # rating for every unrated movie
    predictions = [
        (movie_id, model.predict(user_id, movie_id).est)
        for movie_id in unrated_movies
    ]

    # Top N
    top_n = sorted(predictions, key=lambda x: x[1], reverse=True)[:n]

    result = pd.DataFrame(top_n, columns=['movieId', 'predicted_rating'])
    result = result.merge(movies_df[['movieId', 'title']], on='movieId')
    return result[['title', 'predicted_rating']]

# 
sample_user = filtered['userId'].iloc[0]
recommendations = get_top_n_recommendations(model, sample_user, filtered, movies, n=10)
print(f"Top 10 recommendations for user {sample_user}:")
recommendations

Top 10 recommendations for user 20327:


,title,predicted_rating
0,Blue Planet II (2017),4.734065
1,Planet Earth II (2016),4.728406
2,"Mirror, The (Zerkalo) (1975)",4.714885
3,Shoah (1985),4.708425
4,"Godfather: Part II, The (1974)",4.705074
5,"Civil War, The (1990)",4.683100
6,Planet Earth (2006),4.641785
7,Cosmos,4.631785
8,Magnolia (1999),4.630736
9,Band of Brothers (2001),4.618882
